# Doc lien tuc de lay tin hieu tu Redis va vao lenh

# Vao lenh

In [1]:
import MetaTrader5 as mt5
login = 6338576 # OANDa
password = '123456Qaz1!'
server = 'OANDA-Demo-1'    

import MetaTrader5 as mt
def orderForexSell(data):
    if not mt.initialize(login=login, password=password, server=server):
        print("Initialize() failed, error code =", mt.last_error())
        quit()
    else:
        print("Kết nối thành công với MT5")

    # Dat lenh
    lot = 0.01  # Số lượng lô mà bạn muốn mua
    symbol = data['Symbol']

    if not mt5.symbol_select(symbol, True): # 
        print(f"Failed to select {symbol}, error code =", mt5.last_error())
        quit()

    symbol_info = mt5.symbol_info(symbol)
    if symbol_info is None:
        print(f"{symbol} not found")           
   
    point = symbol_info.point
    price = mt5.symbol_info_tick(symbol).bid  # Sử dụng giá bid cho lệnh bán
    deviation = 20  # Độ lệch giá cho phép

    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": lot,
        "type": mt5.ORDER_TYPE_SELL,
        "price": price,
        "sl": price + 0.02,  # Lệnh dừng lỗ (Stop Loss) cho lệnh bán
        "tp": price - 0.01,  # Lệnh chốt lời (Take Profit) cho lệnh bán
        "deviation": deviation,
        "magic": 234000,
        "comment": "ABC",
        "type_time": mt5.ORDER_TIME_GTC,
        # "type_filling": mt5.ORDER_FILLING_IOC,
        "type_filling": mt5.ORDER_FILLING_FOK,
        # ORDER_FILLING_RETURN 
    }

    result = mt5.order_send(request)
    if result.retcode != mt5.TRADE_RETCODE_DONE:
        print("Failed to send order :", result.retcode, result._asdict())
    else:
        print("Order placed SELL successfully!")
        print(result)

    # Đóng kết nối với MT5
    mt5.shutdown()


In [2]:
def orderForexBuy(data):
    print("Buy")

# Doc tu Redis

In [3]:
import redis
from datetime import datetime

def scan_marketForexRedis():
    r = redis.Redis(host='localhost', port=6379, db=0)
    hash_key = 'OG_Bai tap 9.4_Chien luoc tai lop'
    data = r.hgetall(hash_key)
    # print("Chay luc: ", datetime.now())
    print("Kieu du lieu data: ", type(data))
    
    # Check if data exists
    if not data:
        print("Khong co du lieu trong Redis")
        return
    
    # Convert byte strings to regular strings
    data = {key.decode('utf-8'): value.decode('utf-8') for key, value in data.items()}
    print("Du lieu sau khi decode:", data.keys())  # Debug: show available keys
    
    # Dat lenh giao dich, voi data la dict
    # Check if Buy_Signal key exists and has the right value
    if 'Buy_Signal' in data and data['Buy_Signal'] == "True":
        print("Co tin hieu Buy - dat lenh")
        orderForexBuy(data)
    elif 'Sell_Signal' in data and data['Sell_Signal'] == "True":
        print("Co tin hieu Sell - dat lenh")
        orderForexSell(data)

        # Xoa redis
        r.delete(hash_key)
    else:
        print("Khong co tin hieu giao dich")

# Doc lien tuc

In [ ]:
import schedule

# 1s chay 1 lan
schedule.every(5).seconds.do(scan_marketForexRedis)

while True:
    schedule.run_pending() # Quan doc

Kieu du lieu data:  <class 'dict'>
Du lieu sau khi decode: dict_keys(['Datetime', 'Symbol', 'Insertdate', 'Open', 'High', 'Low', 'Close', 'Volume', 'macd', 'macd_signal', 'macd_hist', 'rsi', 'Buy_Signal', 'Sell_Signal'])
Co tin hieu Sell - dat lenh
Kết nối thành công với MT5
Order placed SELL successfully!
OrderSendResult(retcode=10009, deal=31049719, order=28667217, volume=0.01, price=1.16366, bid=1.16366, ask=1.16376, comment='Request executed', request_id=7487416, retcode_external=0, request=TradeRequest(action=1, magic=234000, order=0, symbol='EURUSD.sml', volume=0.01, price=1.16366, stoplimit=0.0, sl=1.18366, tp=1.15366, deviation=20, type=1, type_filling=0, type_time=0, expiration=0, comment='ABC', position=0, position_by=0))
Kieu du lieu data:  <class 'dict'>
Khong co du lieu trong Redis
Kieu du lieu data:  <class 'dict'>
Khong co du lieu trong Redis
Kieu du lieu data:  <class 'dict'>
Khong co du lieu trong Redis
Kieu du lieu data:  <class 'dict'>
Khong co du lieu trong Redis
Ki